# VQA CAA application evaluation

## Gemma 3

### defs

In [2]:
# !pip install git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3 

In [3]:
import torch
import torch.nn.functional as F
from PIL import Image
from transformers.models.gemma3 import Gemma3Processor, Gemma3ForConditionalGeneration
import json
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm
import kagglehub
from pathlib import Path
from os.path import join

In [4]:
GEMMA_PATH = kagglehub.model_download("google/gemma-3/transformers/gemma-3-4b-it")

In [5]:
proc = Gemma3Processor.from_pretrained(GEMMA_PATH)
mod = Gemma3ForConditionalGeneration.from_pretrained(GEMMA_PATH, torch_dtype=torch.bfloat16).to('cuda')
print(mod)

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi

In [6]:
def make_q(img_pth, q_txt):
  image = Image.open(img_pth).convert("RGB")
  messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": q_txt}
        ]
    }
  ]
  return messages

In [7]:
def generate_a(messages, processor, model):
  inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
  ).to(model.device)

  with torch.inference_mode():
    outputs = model(
        **inputs,
        max_new_tokens=256,
        do_sample=False  # Set to True if you want more creative/varied responses
    )
  generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
  decoded_output = processor.decode(generated_ids, skip_special_tokens=True)

  return decoded_output

In [8]:
def print_answ(img_pth, q_txt):
  print(generate_a(make_q(img_pth, q_txt), proc, mod))

## CAA vect application

### 0. prepare steering vectors

done in *VQA_CAA_create.ipynb*

### 1. src data

In [9]:
path = '/kaggle/input/datasets/timoboz/clevr-dataset'
print("Path to dataset files:", path)
img_dir = join(path, 'CLEVR_v1.0/images/val')

Path to dataset files: /kaggle/input/datasets/timoboz/clevr-dataset


### 2. The Evaluation Dataset Class

In [10]:
class CLEVRCAAEvaluationDataset(Dataset):
    def __init__(self, jsonl_path, image_dir):
        print(f"Loading official benchmark from {jsonl_path}...")
        self.data = []
        print(f"Loading extraction pairs from {jsonl_path}...")
        with open(jsonl_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read().lstrip('\ufeff')

        decoder = json.JSONDecoder()
        idx = 0
        while idx < len(content):
            next_brace = content.find('{', idx)
            if next_brace == -1: break
            try:
                obj, new_idx = decoder.raw_decode(content, next_brace)
                self.data.append(obj)
                idx = new_idx
            except json.JSONDecodeError:
                idx = next_brace + 1

        self.image_dir = Path(image_dir)
        print(f"Successfully loaded {len(self.data)} validation questions.")

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        q = self.data[idx]
        image_path = self.image_dir / q['image_path']
        image = Image.open(image_path).convert('RGB')
        true_answer = str(q['true_answer'])
        q_type = "count" if true_answer.isdigit() else "other"

        return {
            "image": image,
            "question": q['question'],
            "true_answer": true_answer,
            "target_answer": str(q['target_answer']),
            "q_type": q_type
        }

In [11]:
def pil_collate_fn(batch):
    """
    Custom collate function to handle PIL Images.
    Takes a list of dictionaries (the batch) and groups them by key.
    """
    return {key: [item[key] for item in batch] for key in batch[0].keys()}

### 3. Identifying the CLEVR Token IDs

In [12]:
valid_clevr_answers = [str(i) for i in range(10)] # '0' through '9'

allowed_token_ids = []
token_to_string_map = {}

print("Mapping CLEVR answers to Gemma Token IDs:")
for ans in valid_clevr_answers:
    # We encode just the number.
    # add_special_tokens=False prevents adding a <bos> token.
    token_ids = proc.tokenizer.encode(ans, add_special_tokens=False)

    assert len(token_ids) == 1, f"Error: '{ans}' split into multiple tokens!"
    token_id = token_ids[0]

    allowed_token_ids.append(token_id)
    token_to_string_map[token_id] = ans

    print(f"Answer: '{ans}' -> Token ID: {token_id}")

Mapping CLEVR answers to Gemma Token IDs:
Answer: '0' -> Token ID: 236771
Answer: '1' -> Token ID: 236770
Answer: '2' -> Token ID: 236778
Answer: '3' -> Token ID: 236800
Answer: '4' -> Token ID: 236812
Answer: '5' -> Token ID: 236810
Answer: '6' -> Token ID: 236825
Answer: '7' -> Token ID: 236832
Answer: '8' -> Token ID: 236828
Answer: '9' -> Token ID: 236819


### 4. Hook

In [13]:
class SteeringHook:
    def __init__(self, model, steering_vector, target_layer, coefficient):
        self.model = model
        self.steering_vector = steering_vector
        self.target_layer = target_layer
        self.coefficient = coefficient
        self.handle = None

    def _hook_fn(self, module, input, output):
        # 1. Safely locate the hidden states
        if isinstance(output, tuple):
            hidden_states = output[0]
        else:
            hidden_states = output

        # 2. Reshape and move vector to guarantee mathematical alignment
        # steering_vector is (hidden_size,) -> view as (1, 1, hidden_size)
        sv = self.steering_vector.to(hidden_states.device, dtype=hidden_states.dtype)
        if sv.dim() == 1:
            sv = sv.view(1, 1, -1)

        # 3. Modify IN-PLACE!
        # This completely bypasses PyTorch's tuple-returning mechanics,
        # guaranteeing we don't break the model's forward pass.
        hidden_states.add_(self.coefficient * sv)

    def attach(self):
        layer_module = self.model.model.language_model.layers[self.target_layer]
        self.handle = layer_module.register_forward_hook(self._hook_fn)

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

### 5. The Upgraded Evaluation Loop

In [14]:
def get_masked_clevr_prediction(logits, allowed_token_ids, token_to_string_map):
    """Filters the 256k vocab logits down to just the CLEVR valid answers."""
    vocab_size = logits.shape[-1]

    # --- THE FIX: Explicitly set dtype=logits.dtype so it matches Gemma's BFloat16 ---
    mask = torch.full(
        (1, vocab_size),
        float('-inf'),
        device=logits.device,
        dtype=logits.dtype
    )

    allowed_indices = torch.tensor(allowed_token_ids, device=logits.device)
    mask[0, allowed_indices] = logits[0, allowed_indices]

    # Best practice: Upcast to float32 just for the softmax calculation
    # to prevent numerical overflow/underflow, then calculate probabilities.
    probabilities = F.softmax(mask.to(torch.float32), dim=-1)

    predicted_token_id = torch.argmax(probabilities, dim=-1).item()

    prob_dict = {
        token_to_string_map[tid]: probabilities[0, tid].item()
        for tid in allowed_token_ids
    }

    return token_to_string_map[predicted_token_id], prob_dict

In [15]:
def evaluate_steering_vector(model, processor, dataloader, steering_vector, target_layer, multiplier, allowed_token_ids, token_map):
    # 1. Attach the Steering Hook
    hook = SteeringHook(model, steering_vector, target_layer, multiplier)
    hook.attach()
    model.eval()

    metrics = defaultdict(lambda: {"total": 0, "correct_base": 0, "correct_shifted": 0, "p_true": 0.0, "p_target": 0.0})

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Eval Multiplier: {multiplier}"):
            # 1. Grab the raw data from your batch
            q = batch["question"][0]
            image = batch["image"][0]
            true_ans = batch["true_answer"][0]
            target_ans = batch["target_answer"][0]
            q_type = batch["q_type"][0]

            # 2. Build the EXACT same Multimodal Message Array
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": q}
                    ]
                }
            ]

            # 3. Apply the Official Template
            eval_prompt = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            # 3. Forward Pass (Vector is injected automatically by the hook!)
            inputs = processor(text=eval_prompt, images=image, return_tensors="pt").to(model.device)
            outputs = model(**inputs)

            # Get logits for the VERY NEXT token the model wants to predict
            next_token_logits = outputs.logits[:, -1, :]

            # 4. Mask and Predict
            pred_ans, prob_dict = get_masked_clevr_prediction(next_token_logits, allowed_token_ids, token_map)

            # 5. Record Metrics
            stats = metrics[q_type]
            stats["total"] += 1
            stats["p_true"] += prob_dict.get(true_ans, 0.0)
            stats["p_target"] += prob_dict.get(target_ans, 0.0)

            if pred_ans == true_ans:
                stats["correct_base"] += 1
            if pred_ans == target_ans:
                stats["correct_shifted"] += 1

    # 6. Crucial: Remove the hook so it doesn't corrupt future runs
    hook.remove()
    return metrics

### 6. Running the Multiplier Sweep 10

In [1]:
# eval_dataset = CLEVRCAAEvaluationDataset('/kaggle/input/datasets/marekd6/clevr-caas/clevr_caa_extract_val10.jsonl', img_dir)
# eval_loader = DataLoader(eval_dataset, batch_size=1, shuffle=False, collate_fn=pil_collate_fn)

In [2]:
multipliers_to_test = [0.0, 0.5, 1.0, 1.5, 2.0]

In [3]:
# for target_layer in range(15, 13, -1):
#     print(f"\n--- Running Evaluation on layer: {target_layer} ---")
#     steering_vector = torch.load(f"/kaggle/input/datasets/marekd6/clevr-caas/clevr_overcount_plus1_layer{target_layer}100.pt")
#     sweep_results = {}

#     for mult in multipliers_to_test:
#         print(f"\n--- Running Evaluation with Multiplier: {mult} ---")
#         metrics = evaluate_steering_vector(
#             model=mod,
#             processor=proc,
#             dataloader=eval_loader,
#             steering_vector=steering_vector,
#             target_layer=target_layer,
#             multiplier=mult,
#             allowed_token_ids=allowed_token_ids,
#             token_map=token_to_string_map
#         )
#         sweep_results[mult] = metrics

#     print("\n=== FINAL SWEEP RESULTS ===")
#     for mult, metrics in sweep_results.items():
#         print(f"\nMultiplier: {mult}")
#         for q_type, stats in metrics.items():
#             total = stats["total"]
#             if total == 0: continue

#             base_acc = (stats["correct_base"] / total) * 100
#             shift_acc = (stats["correct_shifted"] / total) * 100

#             print(f"  Type: {q_type.ljust(15)} | Base Acc: {base_acc:5.1f}% | Shifted Acc: {shift_acc:5.1f}% | Avg P(Target): {stats['p_target']:5.3f}")

### 6. Running the Multiplier Sweep 100

In [4]:
eval_dataset = CLEVRCAAEvaluationDataset('/kaggle/input/datasets/marekd6/clevr-caas/clevr_caa_extract_val100.jsonl', img_dir)
eval_loader = DataLoader(eval_dataset, batch_size=1, shuffle=False, collate_fn=pil_collate_fn)

Loading official benchmark from /kaggle/input/datasets/marekd6/clevr-caas/clevr_caa_extract_val100.jsonl...
Loading extraction pairs from /kaggle/input/datasets/marekd6/clevr-caas/clevr_caa_extract_val100.jsonl...
Successfully loaded 100 validation questions.


In [5]:
multipliers_to_test = [0.0, 0.5, 1.0, 1.5, 2.0]

In [6]:
for target_layer in range(15, 13, -1):
    print(f"\n--- Running Evaluation on layer: {target_layer} ---")
    steering_vector = torch.load(f"/kaggle/input/datasets/marekd6/clevr-caas/clevr_overcount_plus1_layer{target_layer}100.pt")
    sweep_results = {}

    for mult in multipliers_to_test:
        print(f"\n--- Running Evaluation with Multiplier: {mult} ---")
        metrics = evaluate_steering_vector(
            model=mod,
            processor=proc,
            dataloader=eval_loader,
            steering_vector=steering_vector,
            target_layer=target_layer,
            multiplier=mult,
            allowed_token_ids=allowed_token_ids,
            token_map=token_to_string_map
        )
        sweep_results[mult] = metrics

    print("\n=== FINAL SWEEP RESULTS ===")
    for mult, metrics in sweep_results.items():
        print(f"\nMultiplier: {mult}")
        for q_type, stats in metrics.items():
            total = stats["total"]
            if total == 0: continue

            base_acc = (stats["correct_base"] / total) * 100
            shift_acc = (stats["correct_shifted"] / total) * 100

            print(f"  Type: {q_type.ljust(15)} | Base Acc: {base_acc:5.1f}% | Shifted Acc: {shift_acc:5.1f}% | Avg P(Target): {stats['p_target']:5.3f}")


--- Running Evaluation on layer: 15 ---

--- Running Evaluation with Multiplier: 0.0 ---


Eval Multiplier: 0.0: 100%|██████████| 100/100 [06:59<00:00,  4.20s/it]



--- Running Evaluation with Multiplier: 0.5 ---


Eval Multiplier: 0.5: 100%|██████████| 100/100 [07:03<00:00,  4.23s/it]



--- Running Evaluation with Multiplier: 1.0 ---


...


--- Running Evaluation with Multiplier: 1.5 ---


Eval Multiplier: 1.5: 100%|██████████| 100/100 [07:03<00:00,  4.24s/it]



--- Running Evaluation with Multiplier: 2.0 ---


Eval Multiplier: 2.0: 100%|██████████| 100/100 [07:03<00:00,  4.24s/it]


=== FINAL SWEEP RESULTS ===

Multiplier: 0.0
  Type: count           | Base Acc:  18.0% | Shifted Acc:  16.0% | Avg P(Target): 16.515

Multiplier: 0.5
  Type: count           | Base Acc:  19.0% | Shifted Acc:  19.0% | Avg P(Target): 16.299

Multiplier: 1.0
  Type: count           | Base Acc:  19.0% | Shifted Acc:  18.0% | Avg P(Target): 17.298

Multiplier: 1.5
  Type: count           | Base Acc:  18.0% | Shifted Acc:  16.0% | Avg P(Target): 16.663

Multiplier: 2.0
  Type: count           | Base Acc:  11.0% | Shifted Acc:  15.0% | Avg P(Target): 14.657


### 6. Running the Multiplier Sweep 1k

In [7]:
eval_dataset = CLEVRCAAEvaluationDataset('/kaggle/input/datasets/marekd6/clevr-caas/clevr_caa_extract_val1k.jsonl', img_dir)
eval_loader = DataLoader(eval_dataset, batch_size=1, shuffle=False, collate_fn=pil_collate_fn)

Loading official benchmark from /kaggle/input/datasets/marekd6/clevr-caas/clevr_caa_extract_val1k.jsonl...
Loading extraction pairs from /kaggle/input/datasets/marekd6/clevr-caas/clevr_caa_extract_val1k.jsonl...
Successfully loaded 1000 validation questions.


In [8]:
multipliers_to_test = [0.0, 0.5, 1.0, 1.5, 2.0] 

In [9]:
for target_layer in range(15, 13, -1):
    print(f"\n--- Running Evaluation on layer: {target_layer} ---")
    steering_vector = torch.load(f"/kaggle/input/datasets/marekd6/clevr-caas/clevr_overcount_plus1_layer{target_layer}100.pt")
    sweep_results = {}

    for mult in multipliers_to_test:
        print(f"\n--- Running Evaluation with Multiplier: {mult} ---")
        metrics = evaluate_steering_vector(
            model=mod,
            processor=proc,
            dataloader=eval_loader,
            steering_vector=steering_vector,
            target_layer=target_layer,
            multiplier=mult,
            allowed_token_ids=allowed_token_ids,
            token_map=token_to_string_map
        )
        sweep_results[mult] = metrics
        # print(metrics)
        # for 

    print("\n=== FINAL SWEEP RESULTS ===")
    for mult, metrics in sweep_results.items():
        print(f"\nMultiplier: {mult}")
        for q_type, stats in metrics.items():
            total = stats["total"]
            if total == 0: continue

            base_acc = (stats["correct_base"] / total) * 100
            shift_acc = (stats["correct_shifted"] / total) * 100

            print(f"  Type: {q_type.ljust(15)} | Base Acc: {base_acc:5.1f}% | Shifted Acc: {shift_acc:5.1f}% | Avg P(Target): {stats['p_target']:5.3f}")


--- Running Evaluation on layer: 15 ---

--- Running Evaluation with Multiplier: 0.0 ---


Eval Multiplier: 0.0:   5%|▌         | 50/1000 [03:35<1:08:12,  4.31s/it]


KeyboardInterrupt: 